# MPMAvatar — SDS Physics Hypothesis Test
**Lightning AI Studio Notebook**

Pipeline:
1. Clone repo from GitHub
2. Install dependencies + build CUDA extensions
3. Download Wan 2.2 I2V checkpoint
4. Upload / mount your trained Gaussians
5. Run `train_sds_physics.py`

---
**GPU required:** A100 / A10G (40GB+) recommended for Wan 5B

## Cell 0 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPU      : {torch.cuda.get_device_name(0)}")
print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Set CUDA arch for custom extension builds
import os
cc = torch.cuda.get_device_capability()
arch = f"{cc[0]}{cc[1]}"
os.environ['TORCH_CUDA_ARCH_LIST'] = f"{cc[0]}.{cc[1]}"
os.environ['FORCE_CUDA'] = '1'
print(f"CUDA arch: sm_{arch}  →  TORCH_CUDA_ARCH_LIST={os.environ['TORCH_CUDA_ARCH_LIST']}")

## Cell 1 — Clone Repo

In [ ]:
import os

# ── CHANGE THESE ──────────────────────────────────────────────────────────────
GITHUB_USER  = "YOUR_GITHUB_USERNAME"       # e.g. "poudelsaroj"
GITHUB_REPO  = "MPMAvatar"                  # repo name (without .git)
GITHUB_TOKEN = ""                           # GitHub Personal Access Token
#   → Settings → Developer settings → Personal access tokens → Fine-grained
#   → Repo access: Contents (read)
# ─────────────────────────────────────────────────────────────────────────────

REPO_DIR = f"/teamspace/studios/this_studio/{GITHUB_REPO}"

if not os.path.exists(REPO_DIR):
    if GITHUB_TOKEN:
        clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    else:
        clone_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    print(f"Cloning from github.com/{GITHUB_USER}/{GITHUB_REPO} ...")
    !git clone {clone_url} {REPO_DIR}
else:
    print(f"Repo already at {REPO_DIR}, pulling...")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls

## Cell 2 — Install Python Dependencies

In [ ]:
# Core dependencies — numpy>=1.26 required for Python 3.12+
!pip install -q \
    "numpy>=1.26.0" \
    scipy \
    Pillow \
    tqdm \
    pyyaml \
    wandb \
    einops \
    plyfile \
    trimesh \
    smplx \
    roma \
    jaxtyping \
    mediapy \
    imageio \
    safetensors \
    accelerate

# Warp (MPM solver)
!pip install -q "warp-lang>=1.0.0"

# Transformers + HuggingFace
!pip install -q transformers sentencepiece huggingface_hub

import numpy as np, sys
print(f"numpy {np.__version__} / Python {sys.version.split()[0]}")
print("Done. Run the next cell to install pytorch3d.")

In [ ]:
## Cell 2b — Install pytorch3d (builds from source, ~10-15 min, visible output)
import os
os.environ['FORCE_CUDA'] = '1'
os.environ['TORCH_CUDA_ARCH_LIST'] = '9.0'  # H100

# Install build deps first
!pip install ninja fvcore iopath --quiet

# Build pytorch3d from source with full output so we can see progress
!pip install "git+https://github.com/facebookresearch/pytorch3d.git" \
    --no-build-isolation

try:
    import pytorch3d
    print(f"✓ pytorch3d {pytorch3d.__version__}")
except ImportError as e:
    print(f"✗ pytorch3d FAILED: {e}")

## Cell 3 — Build CUDA Extensions

In [ ]:
import os, sys

REPO_DIR = "/teamspace/studios/this_studio/mpm_avatar_vds"
EXT_DIR  = "/teamspace/studios/this_studio/ext"
os.makedirs(EXT_DIR, exist_ok=True)

# ── diff-gaussian-rasterization ───────────────────────────────────────────────
diff_gauss_dir = f"{EXT_DIR}/diff-gaussian-rasterization"
if not os.path.exists(diff_gauss_dir):
    !git clone --recursive \
        https://github.com/slothfulxtx/diff-gaussian-rasterization.git \
        {diff_gauss_dir}

# Patch rasterizer_impl.h: add #include <cstdint> if missing
impl_h = f"{diff_gauss_dir}/cuda_rasterizer/rasterizer_impl.h"
!head -1 {impl_h}
!grep -q "cstdint" {impl_h} || sed -i '1s/^/#include <cstdint>\n/' {impl_h}
print("After patch, first 3 lines:")
!head -3 {impl_h}

# Build using python setup.py directly (no pip isolation)
!cd {diff_gauss_dir} && python setup.py build_ext --inplace 2>&1 | tail -5
!cd {diff_gauss_dir} && python setup.py install 2>&1 | tail -5
print("diff-gaussian-rasterization done")

# ── simple-knn ────────────────────────────────────────────────────────────────
simple_knn_dir = f"{EXT_DIR}/simple-knn"
if not os.path.exists(simple_knn_dir):
    !git clone https://gitlab.inria.fr/bkerbl/simple-knn.git {simple_knn_dir}

!cd {simple_knn_dir} && python setup.py install 2>&1 | tail -5
print("simple-knn done")

# ── Verify ────────────────────────────────────────────────────────────────────
import importlib, subprocess
subprocess.run([sys.executable, "-c", "import diff_gauss; print('✓ diff_gauss OK')"])
subprocess.run([sys.executable, "-c", "import simple_knn; print('✓ simple_knn OK')"])

## Cell 4 — Install Wan 2.2 Repo

In [ ]:
import os, sys

WAN_REPO_DIR = "/teamspace/studios/this_studio/Wan2.2"

if not os.path.exists(WAN_REPO_DIR):
    !git clone https://github.com/Wan-Video/Wan2.2.git {WAN_REPO_DIR}

# Install all deps
!pip install -q easydict ftfy dashscope diffusers accelerate decord tokenizers imageio-ffmpeg

# flash_attn: download prebuilt wheel directly (avoids cross-device link error)
FLASH_WHL = "flash_attn-2.8.3+cu12torch2.8cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
FLASH_PATH = f"/tmp/{FLASH_WHL}"
if not os.path.exists(FLASH_PATH):
    FLASH_URL = f"https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/{FLASH_WHL}"
    !wget -q {FLASH_URL} -O {FLASH_PATH}
!pip install -q {FLASH_PATH} --no-deps || echo "flash_attn install failed — continuing without it"

# Install Wan itself
!cd {WAN_REPO_DIR} && pip install -e . --no-build-isolation --no-deps -q

if WAN_REPO_DIR not in sys.path:
    sys.path.insert(0, WAN_REPO_DIR)

# Verify — keep importing until no more missing modules
try:
    from wan.modules.model import WanModel
    from wan.modules.vae2_1 import Wan2_1_VAE
    from wan.modules.t5 import T5EncoderModel
    print("✓ Wan modules importable")
except ImportError as e:
    print(f"✗ Wan import failed: {e}")
    print("Run: !pip install <missing_module> then re-run this cell")

## Cell 5 — Download Wan 2.2 I2V Checkpoint

> ⚠️ This is **~40 GB**. Run once, stored persistently in Lightning teamspace.
> Skip if already downloaded.

In [ ]:
import os
from huggingface_hub import snapshot_download, login

HF_TOKEN = "REDACTED_HF_TOKEN"
login(token=HF_TOKEN, add_to_git_credential=False)

WAN_CKPT_DIR = "/teamspace/studios/this_studio/wan_checkpoints_22"
os.makedirs(WAN_CKPT_DIR, exist_ok=True)

if not os.path.exists(f"{WAN_CKPT_DIR}/low_noise_model"):
    print("Downloading Wan2.2-I2V-A14B (~126GB)...")
    snapshot_download(
        repo_id="Wan-AI/Wan2.2-I2V-A14B",
        local_dir=WAN_CKPT_DIR,
        ignore_patterns=["*.md", "*.txt"],
        token=HF_TOKEN,
    )
    print("Done.")
else:
    print("Already downloaded.")

!ls {WAN_CKPT_DIR}/

## Cell 6 — Pre-download T5 Tokenizer

In [ ]:
T5_CACHE = "/teamspace/studios/this_studio/t5_cache"
os.makedirs(T5_CACHE, exist_ok=True)
os.environ['TRANSFORMERS_CACHE'] = T5_CACHE
os.environ['HF_HOME']            = T5_CACHE

from transformers import AutoTokenizer
print("Downloading umt5-xxl tokenizer (~2GB)...")
AutoTokenizer.from_pretrained('google/umt5-xxl', cache_dir=T5_CACHE)
print("Tokenizer ready.")

## Cell 7 — Upload Trained Gaussians

Upload your trained Gaussian model from Stage 1 (appearance training).

**Option A:** Upload via Lightning Studio file browser (drag and drop)

**Option B:** Upload from local machine using the cell below

Expected structure:
```
/teamspace/studios/this_studio/gaussians/
├── point_cloud/
│   └── iteration_XXXXX/
│       └── point_cloud.ply
├── verts_offset.npy
├── cams.npz
└── shadow_net.pt
```

In [ ]:
import os

!pip install -q gdown

PRETRAINED_DIR = "/teamspace/studios/this_studio/pretrained_models"
os.makedirs(PRETRAINED_DIR, exist_ok=True)

GDRIVE_FOLDER_ID = "1ZylC3f8b3wg6Ae4MkVeHFodJc0OVuJfu"

# --remaining-ok skips folders with >50 files (aomap PNGs) but gets all key files
!gdown --folder https://drive.google.com/drive/folders/{GDRIVE_FOLDER_ID} \
    -O {PRETRAINED_DIR} --remaining-ok

!find {PRETRAINED_DIR} -type f | grep -v ".png" | head -40

## Cell 8 — Upload Dataset Files

Upload your `split_idx.npz` and SMPLX fitted params.

Expected structure:
```
/teamspace/studios/this_studio/data/
├── split_idx.npz
├── a1_s1/
│   └── smplx_fitted/
│       ├── 000460.pth
│       ├── 000461.pth
│       └── ...
└── body_models/
    └── smplx/
```

In [ ]:
DATA_DIR = "/teamspace/studios/this_studio/data"

checks = [
    f"{DATA_DIR}/split_idx.npz",
]
for f in checks:
    status = "✓" if os.path.exists(f) else "✗ MISSING"
    print(f"{status}  {f}")

## Cell 9 — Config

Set all paths here. Everything else uses these variables.

In [ ]:
import os

REPO_DIR    = "/teamspace/studios/this_studio/mpm_avatar_vds"
PRETRAINED  = "/teamspace/studios/this_studio/pretrained_models"

# ── Paths ─────────────────────────────────────────────────────────────────────
TRAINED_MODEL_PATH = f"{PRETRAINED}/model/a1_s1/point_cloud/timestep_030000"
MODEL_PATH         = TRAINED_MODEL_PATH
DATASET_DIR        = f"{PRETRAINED}/output/tracking/a1_s1_460_200"
SPLIT_IDX_PATH     = ""
OUTPUT_DIR         = "/teamspace/studios/this_studio/output"
WAN_CKPT_DIR       = "/teamspace/studios/this_studio/wan_checkpoints_22"   # Wan2.2-I2V-A14B
WAN_REPO_ROOT      = "/teamspace/studios/this_studio/Wan2.2"
SDS_CFG            = f"{REPO_DIR}/bridge_sds/configs/sds_test.yaml"
T5_CACHE           = "/teamspace/studios/this_studio/t5_cache"

# ── Dataset config ─────────────────────────────────────────────────────────────
ACTOR              = 1
SEQUENCE           = 1
TRAIN_FRAME_START  = 460
TRAIN_FRAME_NUM    = 10
VERTS_START_IDX    = 460

# ── Training ───────────────────────────────────────────────────────────────────
ITERATIONS         = 100
WANDB_PROJECT      = "MPMAvatar_SDS"
USE_WANDB          = False
WANDB_API_KEY      = ""

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(REPO_DIR)

checks = {
    "point_cloud.ply":  f"{TRAINED_MODEL_PATH}/point_cloud.ply",
    "cams.npz":         f"{TRAINED_MODEL_PATH}/cams.npz",
    "shadow_net.pt":    f"{TRAINED_MODEL_PATH}/shadow_net.pt",
    "verts_offset.npy": f"{TRAINED_MODEL_PATH}/verts_offset.npy",
    "params_460.npz":   f"{DATASET_DIR}/params_460.npz",
    "Wan low_noise":    f"{WAN_CKPT_DIR}/low_noise_model",
    "Wan high_noise":   f"{WAN_CKPT_DIR}/high_noise_model",
    "Wan VAE":          f"{WAN_CKPT_DIR}/Wan2.1_VAE.pth",
    "Wan T5":           f"{WAN_CKPT_DIR}/models_t5_umt5-xxl-enc-bf16.pth",
}
all_ok = True
for name, path in checks.items():
    ok = os.path.exists(path)
    if not ok: all_ok = False
    print(f"{'✓' if ok else '✗'} {name}")
print()
print("✓ Ready to run" if all_ok else "✗ Fix missing files")

## Cell 10 — WandB Login (optional)

In [ ]:
if USE_WANDB:
    import wandb
    if WANDB_API_KEY:
        wandb.login(key=WANDB_API_KEY)
    else:
        wandb.login()  # interactive prompt
    print("WandB logged in")
else:
    import os
    os.environ['WANDB_MODE'] = 'disabled'
    print("WandB disabled")

## Cell 11 — Verify Everything Before Running

In [ ]:
import os

checks = {
    "Repo":                 REPO_DIR,
    "train_sds_physics.py": f"{REPO_DIR}/train_sds_physics.py",
    "bridge_sds/wan22":     f"{REPO_DIR}/bridge_sds/wan22_i2v_guidance.py",
    "SDS config yaml":      SDS_CFG,
    "Gaussians dir":        TRAINED_MODEL_PATH,
    "Wan ckpt dir":         WAN_CKPT_DIR,
    "Wan repo":             WAN_REPO_ROOT,
    "Dataset dir":          DATASET_DIR,
}

all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    if not exists:
        all_ok = False
    print(f"{'✓' if exists else '✗ MISSING'}  {name}: {path}")

print()
print("✓ All checks passed — ready to run" if all_ok else "✗ Fix missing paths above before running")

## Cell 12 — Run SDS Physics Training

In [ ]:
import os
import sys

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, WAN_REPO_ROOT)

# Set tokenizer cache so it finds the pre-downloaded T5
os.environ['TRANSFORMERS_CACHE'] = T5_CACHE
os.environ['HF_HOME']            = T5_CACHE

wandb_flag = "--use_wandb" if USE_WANDB else ""

cmd = f"""
python train_sds_physics.py \\
    --trained_model_path {TRAINED_MODEL_PATH} \\
    --model_path         {MODEL_PATH} \\
    --dataset_dir        {DATASET_DIR} \\
    --split_idx_path     {SPLIT_IDX_PATH} \\
    --output_dir         {OUTPUT_DIR} \\
    --actor              {ACTOR} \\
    --sequence           {SEQUENCE} \\
    --train_frame_start_num {TRAIN_FRAME_START} {TRAIN_FRAME_NUM} \\
    --verts_start_idx    {VERTS_START_IDX} \\
    --wan_ckpt_dir       {WAN_CKPT_DIR} \\
    --wan_repo_root      {WAN_REPO_ROOT} \\
    --sds_cfg            {SDS_CFG} \\
    --iterations         {ITERATIONS} \\
    --save_name          sds_test \\
    {wandb_flag} \\
    --wandb_project      {WANDB_PROJECT}
""".strip()

print("Running command:")
print(cmd)
print("\n" + "="*60 + "\n")

os.system(cmd)

## Cell 13 — Monitor Output Files

In [ ]:
import os
import pandas as pd

# Find param trajectory CSV
csv_candidates = []
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        if f == "param_trajectory.csv":
            csv_candidates.append(os.path.join(root, f))

if csv_candidates:
    csv_path = csv_candidates[-1]
    print(f"Found: {csv_path}")
    df = pd.read_csv(csv_path)
    print(df.tail(20).to_string())
else:
    print("No param_trajectory.csv found yet")

## Cell 14 — Plot Parameter Trajectories

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

if csv_candidates:
    df = pd.read_csv(csv_candidates[-1])

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle("SDS Physics Parameter Optimization", fontsize=14)

    # Parameters
    axes[0,0].plot(df['step'], df['D'],        'b-o', ms=3); axes[0,0].set_title('Density D')
    axes[0,1].plot(df['step'], df['E_Pa'],     'r-o', ms=3); axes[0,1].set_title('Young\'s E (Pa)')
    axes[0,2].plot(df['step'], df['H'],        'g-o', ms=3); axes[0,2].set_title('Height Scale H')
    axes[1,0].plot(df['step'], df['friction'], 'm-o', ms=3); axes[1,0].set_title('Friction')

    # Loss
    axes[1,1].plot(df['step'], df['sds_loss_base'], 'k-o', ms=3); axes[1,1].set_title('SDS Loss (base)')

    # Gradients
    axes[1,2].plot(df['step'], df['grad_D'], label='D')
    axes[1,2].plot(df['step'], df['grad_E'], label='E')
    axes[1,2].plot(df['step'], df['grad_H'], label='H')
    axes[1,2].set_title('SPSA Gradients')
    axes[1,2].legend()

    for ax in axes.flat:
        ax.set_xlabel('step')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/param_trajectory.png", dpi=150)
    plt.show()
    print(f"Saved to {OUTPUT_DIR}/param_trajectory.png")
else:
    print("No CSV data yet")

## Cell 15 — Show Best Params Found

In [ ]:
import numpy as np
import glob

npz_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/sds_best_param_*.npz", recursive=True))

if npz_files:
    latest = npz_files[-1]
    print(f"Latest checkpoint: {latest}")
    data = np.load(latest)
    print("\nBest parameters found:")
    for k, v in data.items():
        print(f"  {k:12s} = {float(v):.6f}")
else:
    print("No checkpoint files found yet")